# c-PSD / Pore Size Distribution Analysis

このNotebookでは、3D SEM / FIB-SEM由来の3D TIFF画像から作成した孔相マスクを用いて、孔径分布を評価します。前回のNotebookでは、全体空隙率と方向別空隙率を計算しました。今回は、空隙の「量」だけでなく、「孔の大きさ分布」を評価します。

参考論文では、c-PSD法により3D空隙内に入る球の半径から孔径分布を推定しています。本Notebookでは、この考え方をPythonで扱いやすい形にし、距離変換およびlocal thicknessを用いて孔径分布を計算します。

**重要:** 本実装は論文のMünch and Holzerのc-PSD法を参考にした、距離変換 / local thicknessベースの近似実装です。厳密な最大球法、centroid path法、constrictivity評価を再現するものではありません。


## 1. ライブラリ読み込み

`porespy`はoptionalです。インストールされていない場合でも、scipyの距離変換による基本解析は実行できます。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff
from scipy.ndimage import median_filter, distance_transform_edt, zoom
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects, remove_small_holes

try:
    import porespy as ps
    HAS_PORESPY = True
except ImportError:
    HAS_PORESPY = False
    print("porespy is not installed. The notebook will run with scipy distance transform only.")


## 2. パラメータ設定

voxel sizeは孔径をµm単位で評価するために必須です。異方性voxelに対応するため、距離変換では `sampling=(VOXEL_SIZE_Z, VOXEL_SIZE_Y, VOXEL_SIZE_X)` を使用します。


In [ ]:
INPUT_TIF = "data/PP1615/PP1615_1300x650x500_10nm_white_pores.tif"
OUTPUT_DIR = "outputs/cpsd_output"

# ファイル名の通り、このデータは孔が白いので False にします。
PORE_IS_DARK = False

# 入力画像はすでに 0/255 の二値画像なので、デフォルトではフィルタをかけません。
USE_MEDIAN_FILTER = False
MEDIAN_SIZE = 2
USE_MANUAL_THRESHOLD = False
MANUAL_THRESHOLD = 128

# voxel size
# 単位は µm
VOXEL_SIZE_Z = 0.02
VOXEL_SIZE_Y = 0.01
VOXEL_SIZE_X = 0.01

# 3D距離変換はメモリ使用量が大きいため、CPSD計算用に間引きます。
# 1: 元解像度、2: 各方向1/2、4: 各方向1/4、8: 各方向1/8
EDT_STRIDE = 8

# PSD bin設定
RADIUS_BIN_WIDTH_UM = 0.01
MIN_RADIUS_UM = 0.0
MAX_RADIUS_UM = None

# 小さいノイズ孔や穴埋め（孔構造を勝手に変えないため、デフォルトではOFF）
REMOVE_SMALL_OBJECTS = False
MIN_OBJECT_SIZE = 64
REMOVE_SMALL_HOLES_FLAG = False
MIN_HOLE_SIZE = 64

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

input_path = Path(INPUT_TIF)
if not input_path.is_absolute():
    input_path = project_root / input_path
INPUT_TIF = str(input_path)

output_dir = Path(OUTPUT_DIR)
if not output_dir.is_absolute():
    output_dir = project_root / output_dir
output_dir.mkdir(parents=True, exist_ok=True)


## 3. 3D TIFF読み込み、前処理、二値化

TIFFを読み込み、中央スライス、輝度範囲、二値化結果を確認します。`PORE_IS_DARK = True` の場合は低輝度側を孔相、`False` の場合は高輝度側を孔相として扱います。


In [ ]:
img = tiff.imread(INPUT_TIF)
print("shape:", img.shape)
print("dtype:", img.dtype)
print("min:", np.min(img))
print("max:", np.max(img))

z_mid = img.shape[0] // 2
plt.figure(figsize=(6, 6))
plt.imshow(img[z_mid], cmap="gray")
plt.title("Center slice - original")
plt.axis("off")
plt.tight_layout()
plt.savefig(output_dir / "center_slice_original.png", dpi=200)
plt.show()


In [ ]:
if USE_MEDIAN_FILTER:
    img_filt = median_filter(img, size=MEDIAN_SIZE)
else:
    img_filt = img.copy()

if USE_MANUAL_THRESHOLD:
    threshold = MANUAL_THRESHOLD
else:
    threshold = threshold_otsu(img_filt)

print("threshold:", threshold)

if PORE_IS_DARK:
    pore = img_filt < threshold
else:
    pore = img_filt > threshold

if REMOVE_SMALL_OBJECTS:
    pore = remove_small_objects(pore, min_size=MIN_OBJECT_SIZE)

if REMOVE_SMALL_HOLES_FLAG:
    pore = remove_small_holes(pore, area_threshold=MIN_HOLE_SIZE)

solid = ~pore
porosity = pore.mean()
print(f"porosity: {porosity:.6f} ({porosity * 100:.2f} %)")

plt.figure(figsize=(6, 6))
plt.imshow(pore[z_mid], cmap="gray")
plt.title("Center slice - pore mask")
plt.axis("off")
plt.tight_layout()
plt.savefig(output_dir / "center_slice_pore_mask.png", dpi=200)
plt.show()


## 4. 距離変換による基本的な孔径分布

孔相に対して3D距離変換を実施します。`distance_um` は、各孔ボクセルが固体/孔界面からどれだけ離れているかを表す局所半径です。単位はµmです。

**注意:** これは厳密なc-PSDではなく、距離変換に基づく「局所半径分布」です。Münch and Holzerのc-PSD法を参考にした近似実装として解釈してください。


In [ ]:
if not np.any(pore):
    raise ValueError(
        "No pore voxels were detected. Check PORE_IS_DARK and the threshold settings. "
        f"Current threshold={threshold}, PORE_IS_DARK={PORE_IS_DARK}."
    )

pore_for_edt = pore[::EDT_STRIDE, ::EDT_STRIDE, ::EDT_STRIDE]
sampling_for_edt = (
    VOXEL_SIZE_Z * EDT_STRIDE,
    VOXEL_SIZE_Y * EDT_STRIDE,
    VOXEL_SIZE_X * EDT_STRIDE,
)

print("EDT input shape:", pore_for_edt.shape)
print("EDT voxel sampling [um]:", sampling_for_edt)

distance_um = distance_transform_edt(
    pore_for_edt,
    sampling=sampling_for_edt
)

radius_values_um = distance_um[pore_for_edt]
diameter_values_um = 2 * radius_values_um

print("number of pore voxels used for EDT:", radius_values_um.size)
print("max radius [um]:", np.nanmax(radius_values_um))


## 5. ヒストグラムPSDの計算

半径分布と直径分布をヒストグラム化し、各binの孔体積割合および累積孔体積割合を計算します。


In [ ]:
if MAX_RADIUS_UM is None:
    max_radius = np.nanmax(radius_values_um)
else:
    max_radius = MAX_RADIUS_UM

bins_radius = np.arange(MIN_RADIUS_UM, max_radius + RADIUS_BIN_WIDTH_UM, RADIUS_BIN_WIDTH_UM)
hist_radius, edges_radius = np.histogram(radius_values_um, bins=bins_radius)
volume_fraction = hist_radius / hist_radius.sum()
cumulative_volume_fraction = np.cumsum(volume_fraction)
radius_center_um = 0.5 * (edges_radius[:-1] + edges_radius[1:])
diameter_center_um = 2 * radius_center_um

psd_df = pd.DataFrame({
    "radius_um": radius_center_um,
    "diameter_um": diameter_center_um,
    "voxel_count": hist_radius,
    "volume_fraction": volume_fraction,
    "cumulative_volume_fraction": cumulative_volume_fraction
})

psd_df.to_csv(output_dir / "distance_transform_psd.csv", index=False)
psd_df.head()


## 6. 代表値の計算

以下の代表値は、孔相ボクセルを母集団とした空隙体積加重平均・分位点です。つまり、孔相体積に占める各局所半径の寄与に基づく値であり、個々の独立孔を数えた個数平均ではありません。


In [ ]:
mean_radius_um = float(np.mean(radius_values_um))
median_radius_um = float(np.median(radius_values_um))
p10_radius_um, p50_radius_um, p90_radius_um = [float(v) for v in np.percentile(radius_values_um, [10, 50, 90])]

mean_diameter_um = float(np.mean(diameter_values_um))
median_diameter_um = float(np.median(diameter_values_um))
p10_diameter_um, p50_diameter_um, p90_diameter_um = [float(v) for v in np.percentile(diameter_values_um, [10, 50, 90])]

summary_df = pd.DataFrame([{
    "input_file": INPUT_TIF,
    "shape_z": img.shape[0],
    "shape_y": img.shape[1],
    "shape_x": img.shape[2],
    "voxel_size_z_um": VOXEL_SIZE_Z,
    "voxel_size_y_um": VOXEL_SIZE_Y,
    "voxel_size_x_um": VOXEL_SIZE_X,
    "threshold": threshold,
    "pore_is_dark": PORE_IS_DARK,
    "porosity": porosity,
    "mean_radius_um": mean_radius_um,
    "median_radius_um": median_radius_um,
    "p10_radius_um": p10_radius_um,
    "p50_radius_um": p50_radius_um,
    "p90_radius_um": p90_radius_um,
    "mean_diameter_um": mean_diameter_um,
    "median_diameter_um": median_diameter_um,
    "p10_diameter_um": p10_diameter_um,
    "p50_diameter_um": p50_diameter_um,
    "p90_diameter_um": p90_diameter_um,
    "method": "distance_transform_edt",
    "notes": "Distance-transform-based pore radius distribution; inspired by c-PSD but not identical to the original Munch-Holzer implementation."
}])
summary_df.to_csv(output_dir / "cpsd_summary.csv", index=False)
summary_df


## 7. グラフ出力

半径分布、直径分布、累積孔径分布を保存します。参考論文のFig. 4のように、体積分率および累積体積分率として孔径分布を確認できます。


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(psd_df["radius_um"], psd_df["volume_fraction"], width=RADIUS_BIN_WIDTH_UM, align="center")
plt.xlabel("Pore radius [µm]")
plt.ylabel("Volume fraction")
plt.title("Pore radius distribution")
plt.tight_layout()
plt.savefig(output_dir / "pore_radius_distribution.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(psd_df["diameter_um"], psd_df["volume_fraction"], width=2 * RADIUS_BIN_WIDTH_UM, align="center")
plt.xlabel("Pore diameter [µm]")
plt.ylabel("Volume fraction")
plt.title("Pore diameter distribution")
plt.tight_layout()
plt.savefig(output_dir / "pore_diameter_distribution.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(psd_df["radius_um"], psd_df["cumulative_volume_fraction"], marker="o")
plt.xlabel("Pore radius [µm]")
plt.ylabel("Cumulative volume fraction")
plt.ylim(0, 1.02)
plt.title("Cumulative pore size distribution")
plt.tight_layout()
plt.savefig(output_dir / "cumulative_pore_size_distribution.png", dpi=200)
plt.show()


## 8. カラーマップ可視化

中央スライスに対して、距離変換で得られた局所半径を孔相のみカラーマップ表示します。大きな孔がどの領域に存在するかを直感的に確認できます。


In [ ]:
z_mid_edt = pore_for_edt.shape[0] // 2
z_mid_img = min(z_mid_edt * EDT_STRIDE, img.shape[0] - 1)
radius_slice = np.where(pore_for_edt[z_mid_edt], distance_um[z_mid_edt], np.nan)

plt.figure(figsize=(7, 6))
plt.imshow(img[z_mid_img], cmap="gray")
im = plt.imshow(radius_slice, cmap="viridis", alpha=0.85, extent=(0, img.shape[2], img.shape[1], 0))
plt.colorbar(im, label="Local radius [µm]")
plt.title("Center slice local radius map")
plt.axis("off")
plt.tight_layout()
plt.savefig(output_dir / "center_slice_local_radius_map.png", dpi=200)
plt.show()


## 9. optional: PoreSpy local thicknessによるc-PSD近似

`HAS_PORESPY = True` の場合のみ、`porespy.filters.local_thickness` を使ってlocal thickness解析を実施します。これは単純な距離変換ではなく、最大内接球に近い考え方で孔径分布を評価するための近似です。

注意点:
- PoreSpy local_thicknessはvoxel単位の等方voxelを前提にする場合があります。
- voxel sizeがZとXYで異なる場合は、前処理で等方voxelへリサンプリングする方が望ましいです。
- 本Notebookでは、まず簡易実装として `min(voxel size)` による換算または等方voxel前提で実施します。
- 厳密にやる場合は、画像を等方voxelにリサンプリングしてからlocal thicknessを実行してください。


In [ ]:
if HAS_PORESPY:
    try:
        lt = ps.filters.local_thickness(pore)
        lt_radius_um = lt[pore] * min(VOXEL_SIZE_Z, VOXEL_SIZE_Y, VOXEL_SIZE_X)

        lt_hist, lt_edges = np.histogram(lt_radius_um, bins=bins_radius)
        lt_volume_fraction = lt_hist / lt_hist.sum()
        lt_cumulative = np.cumsum(lt_volume_fraction)
        lt_radius_center_um = 0.5 * (lt_edges[:-1] + lt_edges[1:])

        lt_df = pd.DataFrame({
            "radius_um": lt_radius_center_um,
            "diameter_um": 2 * lt_radius_center_um,
            "voxel_count": lt_hist,
            "volume_fraction": lt_volume_fraction,
            "cumulative_volume_fraction": lt_cumulative
        })
        lt_df.to_csv(output_dir / "porespy_local_thickness_distribution.csv", index=False)

        plt.figure(figsize=(7, 5))
        plt.bar(lt_df["radius_um"], lt_df["volume_fraction"], width=RADIUS_BIN_WIDTH_UM, align="center")
        plt.xlabel("Pore radius [µm]")
        plt.ylabel("Volume fraction")
        plt.title("PoreSpy local thickness distribution")
        plt.tight_layout()
        plt.savefig(output_dir / "porespy_local_thickness_distribution.png", dpi=200)
        plt.show()

        lt_slice = np.where(pore[z_mid], lt[z_mid] * min(VOXEL_SIZE_Z, VOXEL_SIZE_Y, VOXEL_SIZE_X), np.nan)
        plt.figure(figsize=(7, 6))
        plt.imshow(img[z_mid], cmap="gray")
        im = plt.imshow(lt_slice, cmap="magma", alpha=0.85)
        plt.colorbar(im, label="Local thickness radius [µm]")
        plt.title("PoreSpy local thickness map")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(output_dir / "porespy_local_thickness_map.png", dpi=200)
        plt.show()
    except Exception as e:
        print("PoreSpy local_thickness failed:", e)
else:
    print("PoreSpy local_thickness skipped because porespy is not installed.")


## 10. 等方voxelへのリサンプリング関数

異方性voxelを等方voxelへリサンプリングするための関数です。デフォルトでは実行しません。PoreSpy local thicknessを厳密に扱いたい場合や、Z方向とXY方向のvoxel sizeが大きく異なる場合に検討してください。


In [ ]:
def resample_to_isotropic(binary_volume, voxel_size_zyx, target_voxel_size=None):
    """
    binary_volume: 3D boolean array, shape = (Z, Y, X)
    voxel_size_zyx: tuple of voxel sizes in µm, (z, y, x)
    target_voxel_size: target isotropic voxel size in µm.
                       If None, use min(voxel_size_zyx).
    """
    if target_voxel_size is None:
        target_voxel_size = min(voxel_size_zyx)

    zoom_factors = tuple(v / target_voxel_size for v in voxel_size_zyx)
    resampled = zoom(binary_volume.astype(np.uint8), zoom=zoom_factors, order=0).astype(bool)
    return resampled, target_voxel_size


## 11. 解釈

距離変換ベースの孔径分布を解釈します。結果は、二値化しきい値、voxel size、画像解像度、前処理条件に強く依存します。


In [ ]:
print("=== Interpretation ===")
print(f"平均孔半径: {mean_radius_um:.4f} µm")
print(f"中央値孔半径: {median_radius_um:.4f} µm")
print(f"P10/P50/P90 孔半径: {p10_radius_um:.4f}, {p50_radius_um:.4f}, {p90_radius_um:.4f} µm")
print("この孔径分布は、二値化された3D孔相に基づく距離変換ベースの推定値です。")
print("しきい値、voxel size、画像解像度、前処理条件によって結果は変化します。")
print("特に解像度以下の微細孔や、FIB-SEMで見えないナノフィブリルは評価に反映されません。")
